# 04｜官方 Colab 流水线 × H1 本地评分

Arc 官方 [STATE for Virtual Cell Challenge Colab](https://colab.research.google.com/drive/1QKOtYP7bMpdgDJEipDxaJqOchv7oQ-_l)（VC2025 版）演示了 State 在 H1 数据上“下载 → 训练 → 推理 → 打包”的完整流程；本仓库 `references/vcc2026-h1-benchmark` 提供在**同一份 H1 训练数据**上、按 2026 年六指标口径的本地评分。本课把两者接成一条可复现的开发流水线：一份训练产出，两处出口。

**学习目标：** 说出 Colab 22 个 cell 各做什么、对应本地哪份资料；复现 benchmark 的 126 靶点面板并生成推理 TSV；分清哪一步必须换掉 Colab 的默认推理目标；守住防泄漏与文件分离两条边界。

**运行方式：** 本课在 Python 3.13 学习环境本地运行，只做哈希校验、靶点筛选与 TSV 生成；不联网下载大资产、不安装 arc-state、不训练、不评分。真正的训练与评分命令在单元四，需另建 Python 3.12 环境和 GPU。上一课的算力结论（单 T4 约 9 小时/run）直接适用。

## 单元一：官方 Colab 与固定副本

**Colab 是可变文档**：作者随时可能改动。源码审计（[state-training-source-audit](../docs/research/state-training-source-audit.md)）通过 Drive 下载入口读取了当时的 22 个 cell，并记录 SHA-256 `0b3888b9…`。本仓库的 `notebook/90_ref_arc_vcc2025_colab_official.ipynb` 是该版本的**逐字节固定副本**——本课所有 cell 序号都以它为准。

重新执行本课前先校验哈希：若 Arc 更新了 Colab，副本哈希会变，cell 序号和内容都可能移动，需要重新审计后再更新本课的映射表，不能沿用旧序号。

In [1]:
import csv, hashlib, json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pyproject.toml').exists()
             and (p / 'docs/Official-website').is_dir()), None)
if ROOT is None:
    raise RuntimeError('请在 virtual-cell-2026 仓库内启动 notebook')
COLAB = ROOT / 'notebook/90_ref_arc_vcc2025_colab_official.ipynb'
ASSETS = ROOT / 'notebook/assets/h1-benchmark'
OUT = ROOT / 'output/notebook-learning/04-bridge'
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA = '0b3888b9a36e6fbfc056b9e5585d825aa5a97a92f34d3bc2ff69cba064f21422'
actual_sha = hashlib.sha256(COLAB.read_bytes()).hexdigest()
if actual_sha != EXPECTED_SHA:
    raise AssertionError(
        f'Colab 副本哈希 {actual_sha} 与审计版本不一致；'
        '请先对照 docs/research/state-training-source-audit.md 重新审计，再更新本课映射表')
colab = json.loads(COLAB.read_text())
print(f'官方 Colab 副本：{len(colab["cells"])} 个 cell | SHA-256 校验通过')

官方 Colab 副本：22 个 cell | SHA-256 校验通过


In [2]:
ROLES = {
    0: ('标题：STATE for Virtual Cell Challenge', '—'),
    1: ('流程总览：训练 → 推理 → 提交', '本课单元二'),
    2: ('内嵌截图（base64 PNG）', '—'),
    3: ('“克隆仓库”标题', '—'),
    4: ('git clone ArcInstitute/state 并安装', '教程 §7 环境与目录'),
    5: ('训练数据说明（VC2025 support set）', '教程 §4 数据下载单'),
    6: ('内嵌截图 fewshot', '—'),
    7: ('鼓励替换/追加训练数据集', '教程 §4.3 暂缓的数据'),
    8: ('“下载数据”标题', '—'),
    9: ('下载 competition_support_set.zip（Arc 公共 GCS）', 'vcc-h1 setup 下载同一 H1 训练对象'),
    10: ('解压 support set', '同上'),
    11: ('设置 W&B entity（实验追踪）', '离线训练可关闭'),
    12: ('“安装与训练”标题', '—'),
    13: ('安装并验证 state 命令行', '教程 §7.1'),
    14: ('state tx train：40k 步、starter.toml、ESM2 靶点特征', '教程 §8 原生微调入口'),
    15: ('“推理”标题', '本课单元三/四'),
    16: ('列出训练产生的 checkpoints', '教程 §8.1 init_from 与 resume'),
    17: ('state tx infer：对 VC2025 验证模板推理', '桥接改动点：换 H1 126 靶点 TSV'),
    18: ('“cell-eval 与提交”标题', '官方提交才需要'),
    19: ('安装 zstd', '—'),
    20: ('cell-eval prep 打包 prediction.vcc', '开发集改用 vcc-h1 score'),
    21: ('完成提示：上传排行榜', 'H1 分数只作本地开发'),
}
rows = []
for i, cell in enumerate(colab['cells']):
    what, local = ROLES.get(i, ('（未标注）', '—'))
    rows.append({'colab_cell': i, '类型': cell['cell_type'], '内容': what, '本地对应': local})
display(pd.DataFrame(rows))

,colab_cell,类型,内容,本地对应
0,0,markdown,标题：STATE for Virtual Cell Challenge,—
1,1,markdown,流程总览：训练 → 推理 → 提交,本课单元二
2,2,markdown,内嵌截图（base64 PNG）,—
3,3,markdown,“克隆仓库”标题,—
4,4,code,git clone ArcInstitute/state 并安装,教程 §7 环境与目录
5,5,markdown,训练数据说明（VC2025 support set）,教程 §4 数据下载单
6,6,markdown,内嵌截图 fewshot,—
7,7,markdown,鼓励替换/追加训练数据集,教程 §4.3 暂缓的数据
8,8,markdown,“下载数据”标题,—
9,9,code,下载 competition_support_set.zip（Arc 公共 GCS）,vcc-h1 setup 下载同一 H1 训练对象


## 单元二：两条流水线共用一份 H1 数据

Colab 的 `competition_support_set.zip` 与 benchmark 的 `vcc-h1 setup` 都指向 Arc 公共 GCS 上的 **2025 H1 训练对象**（约 15.48 GB，注册表记录其 SHA-256）。所以**训练这一步完全共用**：一个按 Colab 训出的模型，既服务官方提交路线，也服务本地评分路线。两条流水线从**推理**开始分歧：

| 步骤 | Colab 原生（VC2025） | 对接 H1 benchmark |
|---|---|---|
| 数据 | `competition_support_set.zip`（H1 训练对象 + `starter.toml` + ESM2 特征 + 验证模板） | `vcc-h1 setup` 下载同一 H1 训练对象 + 固定参考资产（registry 校验 SHA-256） |
| 训练 | `state tx train`，40k 步 | 完全相同，一份模型两处使用 |
| 推理目标 | `competition_val_template.h5ad`（VC2025 验证靶点） | 126 个 benchmark 靶点 × 400 细胞的 TSV |
| 推理输入 | 模板自带的支持细胞 | H1 的 38,176 个 NTC 对照细胞（18,080 基因轴） |
| 输出 | `competition/prediction.h5ad` | `prediction_h1.h5ad`：50,400 × 18,080 |
| 出口 | `cell-eval prep` 打包 → 官方排行榜 | `vcc-h1 validate` → `score` 六指标（本地开发） |

两份输出**合同不同**，这决定了 Colab 的 `prediction.h5ad` 不能直接喂给 `vcc-h1 score`：

| 合同项 | Colab 输出（VC2025 模板） | vcc-h1 评分合同 |
|---|---|---|
| 行数 | 由模板决定 | 恰好 50,400（126 × 400） |
| 基因轴 | VC2025 面板 | 恰好 18,080 个 H1 基因、顺序固定 |
| `obs` 列 | 模板字段 | 恰好一列 `target_gene` |
| 对照行 | infer 会保留并模拟输入对照行 | 不含任何对照细胞 |
| 数值 | 表达值（log 空间），不是合规计数 | raw 非负整数，每细胞总量 1–1,000,000 |

桥接因此 = **换推理目标 + 归一导出**。另外 lesson 10 §10.1 提醒过：`state tx infer` 写出的是表达值，从浮点表达到合规 raw counts 需要经过校准的计数生成方案，不是 `expm1` 加四舍五入一步了事。

## 单元三：推理目标替换——从 150 个靶点到 126 个

benchmark 的评分面板是 H1 训练数据中**实测细胞数 ≥ 400 的 126 个靶点**（150 个训练靶点中排除 24 个不足 400 细胞的）。官方 `pert_counts_Training.csv` 自带 `n_cells` 列，筛选可以精确复现。本仓库按课程惯例把这份 **2.8 KB 的公开元数据**缓存进 `notebook/assets/h1-benchmark/`，并用 `sources.json` 记录来源——下一格双重校验：缓存文件 ↔ `sources.json` ↔ benchmark 注册表 `benchmark-v1.json`。

推理 TSV 的格式与教程 §11.1 一致：两列 `perturbation`、`num_cells`。正式推理前以 `vcc-h1 setup` 产出的固定参考清单为准——工具在评分时会校验靶点集，本 TSV 只是让推理输入与评分面板对齐。

In [3]:
meta = json.loads((ASSETS / 'sources.json').read_text())
entry = next(f for f in meta['files'] if f['path'] == 'pert_counts_Training.csv')
csv_path = ASSETS / entry['path']
cached_sha = hashlib.sha256(csv_path.read_bytes()).hexdigest()
assert cached_sha == entry['sha256'], f'缓存文件与 sources.json 不一致：{cached_sha}'

registry = json.loads(
    (ROOT / 'references/vcc2026-h1-benchmark/assets/benchmark-v1.json').read_text())
registered_sha = registry['sources']['target_counts']['sha256']
assert registered_sha == cached_sha, '本地缓存与 benchmark 注册表不同源'
print('缓存 CSV ↔ sources.json ↔ benchmark-v1.json 三方 SHA-256 一致')

rows = list(csv.DictReader(csv_path.read_text().splitlines()))
keep = [r for r in rows if int(r['n_cells']) >= 400]
assert len(rows) == 150, f'训练靶点应为 150 个，读到 {len(rows)}'
assert len(keep) == 126, f'n_cells>=400 应为 126 个，得到 {len(keep)}'

tsv_path = OUT / 'h1_infer_targets.tsv'
with tsv_path.open('w') as f:
    f.write('perturbation\tnum_cells\n')
    for r in keep:
        f.write(f"{r['target_gene']}\t400\n")

tsv = pd.read_csv(tsv_path, sep='\t')
assert (tsv.num_cells == 400).all() and tsv.perturbation.is_unique
print(f'150 个训练靶点 → 保留 {len(keep)} 个（排除 {len(rows) - len(keep)} 个 <400 细胞）')
print(f'TSV 写出：{tsv_path.relative_to(ROOT)}，共 {len(tsv)} 行 × 400 细胞')
display(tsv.head(8))

缓存 CSV ↔ sources.json ↔ benchmark-v1.json 三方 SHA-256 一致
150 个训练靶点 → 保留 126 个（排除 24 个 <400 细胞）
TSV 写出：output/notebook-learning/04-bridge/h1_infer_targets.tsv，共 126 行 × 400 细胞


,perturbation,num_cells
0,TMSB4X,400
1,PRCP,400
2,TADA1,400
3,HIRA,400
4,IGF2R,400
5,NCK2,400
6,MED13,400
7,MED12,400


## 单元四：专用环境里的五步命令

以下命令**本地不执行**：需要 GPU、磁盘上的 H1 数据和独立环境。`state` 与 `vcc-h1` 都要求 **Python < 3.13**（后者 `>=3.11,<3.13`），与学习环境分开，推荐一个 Python 3.12 环境装两者；`state` 按 Colab cell 3–4、13 的方式 clone 官方仓库后 `uv sync`，`vcc-h1` 按其 README 用 `uv tool install …@v0.2.0` 固定版本。示例路径沿用教程的 `/data/vcc2026` 风格，可替换。

```bash
# 1) 准备 H1 数据与参考资产（一次；15.48 GB 下载 + 约 16 GiB 磁盘峰值 + 15–25 分钟提取）。
#    若 support set 里已有训练对象，用 --h1 直接提供，避免二次下载；vcc-h1 会按注册表校验。
vcc-h1 setup --h1 /data/vcc2026/support/adata_Training.h5ad \
  --data-dir /data/vcc2026/evaluation/h1-data
vcc-h1 check --data-dir /data/vcc2026/evaluation/h1-data

# 2) 训练：Colab cell 14 原样即可（40k 步，单 T4 实测约 1.25 step/s ≈ 9 小时）。
#    若按教程 §8 做了 init_from/划分调整，保持两路线共用同一份训练产出。

# 3) 桥接推理：TSV 换成上一格生成的 126 靶点；输入用 H1 的 38,176 个 NTC 对照细胞
#    （可用 setup 产出的 control-only H5AD，或自行从训练对象抽取；先核对 raw 整数与 18,080 轴）。
uv run state tx infer \
  --output /data/vcc2026/predictions/prediction_h1_raw.h5ad \
  --model-dir /data/vcc2026/competition/first_run \
  --checkpoint /data/vcc2026/competition/first_run/checkpoints/step=20000.ckpt \
  --adata /data/vcc2026/evaluation/h1-data/h1_ntc_controls.h5ad \
  --pert-col target_gene --control-pert non-targeting \
  --tsv output/notebook-learning/04-bridge/h1_infer_targets.tsv \
  --max-set-len 64 --seed 42

# 4) 归一导出到 benchmark 合同：只留预测行（50,400 × 18,080、obs 仅 target_gene），
#    并用教程 §10 的校准方案生成 raw 整数计数（每细胞总量 1–1,000,000）。

# 5) 校验、评分，与无效应基线对比（作者报告的 unchanged-control 基线约 -0.045，待本地复现）。
vcc-h1 validate /data/vcc2026/predictions/prediction_h1.h5ad \
  --data-dir /data/vcc2026/evaluation/h1-data
vcc-h1 score /data/vcc2026/predictions/prediction_h1.h5ad \
  --data-dir /data/vcc2026/evaluation/h1-data --gene-chunk 512 --de-threads 4 \
  --output /data/vcc2026/evaluation/h1-st-colab
vcc-h1 score-control-baseline --data-dir /data/vcc2026/evaluation/h1-data \
  --output /data/vcc2026/evaluation/h1-control
```

`score` 产出 `scores.csv`（六项缩放指标 + `avg_score`）、`aggregates.csv`、`per_target.csv` 与 `manifest.json`（来源指纹）。评分全程 CPU 分块，不占 GPU；六指标含义见教程 §11.3。

## 单元五：两条边界，一个诊断题

**数据边界（防泄漏）**：Colab 默认在**全量 H1**（含 126 个评分靶点的扰动细胞）上训练。直接评分得到的 `avg_score` 是**内样本开发分**，能验证流水线与格式，不能声称“未见背景泛化”。要作留出声明，需按教程 §5 修改训练划分、把 126 个靶点的扰动细胞从训练中排除，并报告预训练暴露。评分器用的参考 DE、pseudobulk 锚点只进评分器、不进训练器。

**文件边界**：H1 导出（50,400 × 18,080）与官方提交（360,000 × 18,533）是**两个导出器、两次检查**。H1 轴与 2026 轴相差 3 个 H1 特有基因，模型若只建 18,533 轴，需要专门处理映射（教程 §11.2）；H1 评分通过不等于 `vcc prep` 通过。

**诊断题：**Colab cell 17 直接跑出的 `competition/prediction.h5ad`，能直接交给 `vcc-h1 score` 拿六指标吗？

<details><summary>参考答案</summary>不能。它的推理目标是 VC2025 验证模板，不是 126 个 benchmark 靶点；行数、`obs` 结构与 18,080 轴的 50,400 行合同不符；且 infer 原始写出仍是表达值而非合规 raw counts。正确路径是按单元四换 TSV 重新推理，再归一导出后评分。</details>

## 本课与整体路径的连接

至此，03 课的预算表落到了实处：一次训练（单 T4 约 9 小时）+ 一次本地评分（CPU 分块）构成最小的“训练→验证”闭环，也是后续所有微调实验的对照协议起点。

- 教程：[10｜State 微调实战与算力预算](../docs/lessons/10-State微调实战与算力预算.md)（§5 划分、§8 训练、§10 计数生成、§11 评分）
- 审计：[H1 benchmark 核验](../docs/research/h1-benchmark-audit.md)、[State 训练来源审计](../docs/research/state-training-source-audit.md)
- 工具：[references/vcc2026-h1-benchmark](../references/vcc2026-h1-benchmark/README.md)（v0.2.0）

本课只做了哈希校验、靶点面板复现与 TSV 生成；未训练模型、未下载 H1 数据、未执行评分，因此不产生也不暗示任何六指标分数。